In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

In [2]:
orders   = pd.read_csv('C:/Users/shere/OneDrive/Desktop/ecommerce-sales-intelligence/data/raw/olist_orders_dataset.csv')
items    = pd.read_csv('C:/Users/shere/OneDrive/Desktop/ecommerce-sales-intelligence/data/raw/olist_order_items_dataset.csv')
customers= pd.read_csv('C:/Users/shere/OneDrive/Desktop/ecommerce-sales-intelligence/data/raw/olist_customers_dataset.csv')
products = pd.read_csv('C:/Users/shere/OneDrive/Desktop/ecommerce-sales-intelligence/data/raw/olist_products_dataset.csv')
payments = pd.read_csv('C:/Users/shere/OneDrive/Desktop/ecommerce-sales-intelligence/data/raw/olist_order_payments_dataset.csv')
reviews  = pd.read_csv('C:/Users/shere/OneDrive/Desktop/ecommerce-sales-intelligence/data/raw/olist_order_reviews_dataset.csv')

In [3]:
for name, df in {'orders': orders, 'items': items, 
                   'customers': customers, 'products': products,
                   'payments': payments, 'reviews': reviews}.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

orders: 99,441 rows, 8 columns
items: 112,650 rows, 7 columns
customers: 99,441 rows, 5 columns
products: 32,951 rows, 9 columns
payments: 103,886 rows, 5 columns
reviews: 99,224 rows, 7 columns


In [4]:
# What columns do we have in orders?
print("=== ORDERS TABLE ===")
print(orders.info())
print()
print("First 3 rows:")
orders.head(3)

=== ORDERS TABLE ===
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 21.9 MB
None

First 3 rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [5]:
# Check nulls in every table
for name, df in {'orders': orders, 'items': items,
                   'customers': customers, 'products': products,
                   'payments': payments, 'reviews': reviews}.items():
    nulls = df.isnull().sum().sum()
    print(f"{name}: {nulls} total null values")

orders: 4908 total null values
items: 0 total null values
customers: 0 total null values
products: 2448 total null values
payments: 0 total null values
reviews: 145903 total null values


In [6]:
# Step 1: Fix date columns — convert text to actual dates
date_cols = ['order_purchase_timestamp', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# Step 2: Keep only delivered orders (clean data)
orders_clean = orders[orders['order_status'] == 'delivered'].copy()

# Step 3: Add useful time columns
orders_clean['year_month'] = orders_clean['order_purchase_timestamp'].dt.to_period('M').astype('str')
orders_clean['year']       = orders_clean['order_purchase_timestamp'].dt.year
orders_clean['month']      = orders_clean['order_purchase_timestamp'].dt.month

# Step 4: Calculate actual delivery days
orders_clean['delivery_days'] = (
    orders_clean['order_delivered_customer_date'] - 
    orders_clean['order_purchase_timestamp']
).dt.days

print(f"Delivered orders: {len(orders_clean):,}")

Delivered orders: 96,478


In [7]:
# Step 5: Merge all tables together
master = orders_clean     .merge(items,     on='order_id',     how='left')     .merge(payments,  on='order_id',     how='left')     .merge(customers, on='customer_id',  how='left')     .merge(products,  on='product_id',   how='left')     .merge(reviews[  ['order_id','review_score']],
                     on='order_id',     how='left')

print(f"Master dataframe: {master.shape[0]:,} rows, {master.shape[1]} columns")
master.head(3)

Master dataframe: 115,723 rows, 35 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,year_month,year,...,customer_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017-10,2017,...,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,4.0
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017-10,2017,...,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,4.0
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017-10,2017,...,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,4.0


In [8]:
# Step 6: Save master dataset — you'll use this every day this week
master.to_csv('../data/processed/master.csv', index=False)
print("master.csv saved successfully!")
print(f"Columns: {list(master.columns)}")

master.csv saved successfully!
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'year_month', 'year', 'month', 'delivery_days', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'review_score']
